<center><h1>Técnicas de Normalización de Datos</center>

<b>Maestría</b>: Inteligencia Artificial Aplicada <br>
<b>Asignatura:</b> Procesamiento Acelerado de Lenguaje Natural <br>
<b>Profesor:</b> Edwin J. Rueda

## Normalización

La normalización de datos consiste en transformar texto bruto en una representación más adecuada para su análisis. En este *Notebook* abordaremos las siguientes técnicas de normalización o procesamientos de texto:

- **Tokenización**
- **Lematización**
- **Stemming**
- **Lowercasing**
- **Eliminación de Stopwords**

### Tokenización
Tokenizar consiste el dividir texto en palabras o caracteres según sea la tarea que abarquemos. La tokenización se emplea como paso inicial al pre-procesar un texto y nos permite porteriormente:
- Analizar la estructura/morfología de cada palabra.
- Construir vocabularios para psoteriormente aplicar modelos estadisticos/redes neuronales/etc.

$$ tokenizar(\text{El dia está soleado}) = [\text{El, día, está, soleado}]$$

#### ¿ Por qué aplicar la tokenización ?

Tokenizar el corpus/texto, es la fase inicial de todo procesamiento de lenguaje natural. Ya que nuestros modelos próximos en donde utilizaremos nuestro corpus no entienden texto bruto, por lo cual debemos pre-procesar, normalizar/mapear nuestros datos y luego si darlos como entrada a determinado modelo estadístico, red neuronal, etc.

<center><img src="./images/Esquemas-Normalization.png"  height="200"></center>

#### Tokenización con Nalural Language Toolkit (NLTK)

In [1]:
import nltk
nltk.download('punkt_tab') #se descarga el mapeo del tokenizador en español

sentence = "el día en que todo comenzó, fue genial"
tokens = nltk.word_tokenize(sentence, language='spanish')
print(f"Podemos tokenizar textos: {sentence}")
print(f"tokens: {tokens} \n")

[nltk_data] Downloading package punkt_tab to /home/edwin/nltk_data...


Podemos tokenizar textos: el día en que todo comenzó, fue genial
tokens: ['el', 'día', 'en', 'que', 'todo', 'comenzó', ',', 'fue', 'genial'] 



[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


**Nota:** Cómo podemos ver, después de toquenizado un corpus, podemos crear un diccionario de palabras y asignar a cada palabra un escalar, un vector, una matriz, etc. Lo que nos permitirá que nuestros modelos a entrenar, puedan recibir una entrada esperada.

$$map(token) = value$$

### Lematización
La lematización busca reducir una palabra a su forma canónica o raíz, teniendo en cuenta el contexto gramatical (su etiqueta morfosintáctica). A esto, se le conoce como **lema**.

$$Lema(\text{Que tan complejo es lematizar un texto en español}) \rightarrow \text{Que tanto complejo ser lematizar uno texto en español}$$ 

Para este tipo de procesamiento en corpus en español, *nltk* no tiene modelos disponibles. Por lo cual haremos uso de *Spacy*.

##### Proceso de lematización de una palabra
1) Se tokeniza el corpus/texto
2) Se etiqueta morfosintácticamente cada token/palabra del corpus.
3) Se genera el lema correcto en base a una base de datos léxica del modelo lingüistico.

#### ¿ Por qué lematizar ? 

Su principal funcionalidad es reducir la dimensionalidad de nuestro vocabulario para el posterior entrenamiento de modelos.

$$lema([\text{caminar, camina, caminó, caminaría, caminamos}]) \rightarrow \text{caminar}$$

**Nota**: En este ejemplo, reducimos la dimensionalidad de 5 a 1.

#### Lematización con *Spacy*
Utilizaremos un [pipeline](https://spacy.io/models/es) disponibilizado por *spacy*. Este pipeline nos permite realizar los 3 pasos mencionados anteriormente y nos genera un [documento](https://spacy.io/api/doc) con los respectivos **lemas**.

$$pipeline(corpus) \rightarrow Doc$$

##### ¿Qué es un documento en *spacy*?
- Un *Doc* es una clase que contiene una secuencia de **Tokens**. Donde a su vez, cada [token](https://spacy.io/api/token) (palabra, espacio en blanco, puntuación, etc) contiene una estructura individual (atributos) que podemos extraer de dicho token, como su **lema**.

In [ ]:
# Descargamos el pipeline de spacy
!python3 -m spacy download es_core_news_sm

In [3]:
import spacy
#cargamos el pipeline a utilizar
nlp_pipe = spacy.load("es_core_news_sm") 

- Ahora podemos lematizar un corpus/texto

In [7]:
doc = nlp_pipe("Qué tan complejo es lematizar un texto en español")
print(f"doc: {doc} , type: {type(doc)}")
print("------------")
for token in doc:
    print(f"token: {token.text:10} -> lemma: {token.lemma_:10} -> morpho {token.morph.to_dict()}")

doc: Qué tan complejo es lematizar un texto en español , type: <class 'spacy.tokens.doc.Doc'>
------------
token: Qué        -> lemma: qué        -> morpho {'Number': 'Sing', 'PronType': 'Int,Rel'}
token: tan        -> lemma: tanto      -> morpho {}
token: complejo   -> lemma: complejo   -> morpho {'Gender': 'Masc', 'Number': 'Sing'}
token: es         -> lemma: ser        -> morpho {'Mood': 'Ind', 'Number': 'Sing', 'Person': '3', 'Tense': 'Pres', 'VerbForm': 'Fin'}
token: lematizar  -> lemma: lematizar  -> morpho {'VerbForm': 'Inf'}
token: un         -> lemma: uno        -> morpho {'Definite': 'Ind', 'Gender': 'Masc', 'Number': 'Sing', 'PronType': 'Art'}
token: texto      -> lemma: texto      -> morpho {'Gender': 'Masc', 'Number': 'Sing'}
token: en         -> lemma: en         -> morpho {}
token: español    -> lemma: español    -> morpho {'Gender': 'Masc', 'Number': 'Sing'}


### Stemming

Es el proceso por el cual se lleva un token/palabra a su raíz (*stem*). Realmente, su raíz puede que no sea una palabra propiamente, pero podríamos decir que es un "prefijo" de dicha palabra. Es un método muy rápido pero agresivo para normalizar texto.

$$stem(\text{saltar}) \rightarrow \text{salt}$$
$$stem(\text{camina}) \rightarrow \text{camin}$$
$$stem(\text{Que tan complejo es lematizar un texto en español}) \rightarrow ?$$

#### ¿Cómo se saca la raíz de una palabra/token? 
- Se aplican un conjunto de reglas para eliminar los sufijos.
- Este conjunto de reglas no dependen de la categoría gramatical. Son aplicadas a todos los *tokens* por igual.

#### ¿Por qué sacar el *stem*/raíz de una palabra/*token*?
- Es una forma rápida y sencilla de reducir la dimensionalidad de palabras dentro de un corpus y nos permite realizar busquedas rápidas dentro de documentos. No obstante, solo nos sirve en tareas donde realmente no necesitemos una precisión gramatical de las palabras.

#### Implementando Stem con *nltk*

El agoritmo empleado en *nltk* para sacar la raíz de las palabras es el ***Snowball***, nombrado así por su creador. Este [algoritmo](https://snowballstem.org/), consta de una [base de reglas](https://snowballstem.org/algorithms/spanish/stemmer.html) aplicadas para eliminar los sufijos de las palabras, dejando unicamente su raíz. No obstante, no preserva la categoria de la plabra, ni su significado original. Siendo que dos palabras podrían tener la misma raíz.

In [8]:
from nltk.stem.snowball import SnowballStemmer

# definimos el stemmer
stemmer = SnowballStemmer('spanish')
word = "complejo"
print(f"Palabra -> {word} | stem -> {stemmer.stem(word)}")

Palabra -> complejo | stem -> complej


- Podemos definir una función para sacar la raíz de las palabras dentro de una sentencia.

In [9]:
def get_stem_sentence(sentence):
    result = []
    for word in sentence.split():
        result.append(stemmer.stem(word))
    return result

In [70]:
get_stem_sentence("Que tan complejo es lematizar un texto en español")

['que', 'tan', 'complej', 'es', 'lematiz', 'un', 'text', 'en', 'español']

**Nota**: Como podemos observar, sacar la raíz de una palabra es un método que es rápido, ya que implica aplicar una serie de reglas. Pero no mantiene el significado de dicha palabra. En ocasiones, puede provocar que dos palabras terminen teniendo la misma raíz.

### *Lowercasing* y *Stopwords*

*Lowercasing* y *stopwords* son dos conceptos fundamentales en NLP. El primero, es muy esencial y consiste en llevar todas las palabras de un corpus a minúsculas, esto debido a que si generamos tokens con un corpus que contenga mayúsculas, podríamos generar dos o más tokens para una misma palabra.

<center>Ella, ella, ELLA -> son 3 tokens diferentes</center>

Aunque suene lógico siempre llevar los corpus a palabras en minúsculas, para tareas de reconocimiento de identidades, no se debería aplicar *lowercasing*, debido a que los nombres propios comienzan con mayúscula. Entonces sería muy útil conservar estas letras mayúsculas.

Por otro lado, las ***stopwords*** consisten en palabras que aparecen con mucha frecuencia en un determinado idioma y no suelen aportar valor (articulos, preposiciones, adverbios, etc).

#### ¿Por qué eliminar las stopwords?

- Como mencionabamos anteriormente, al igual que en la lematización o stemmer. Eliminar las **stopwords** nos permite reducir la dimensionalidad de nuestro corpus. Lo que nos permitiría mas adelante disminuir el tiempo de entrenamiento de modelos de AI.
- Reducimos ruido de nuestro conjunto de datos (Nota: solo si las  *stopwords* no son necesarias en nuestra tarea).

**Nota:** Tener en cuenta que no siempre es útil eliminar *stopwords*. Por ejemplo, en tareas de análisis de sentimiento, resulta útil mantener el adverbio **No** para saber si a una persona no le gusta determinada cosa.


#### Implementemos *lowercasing* y *stopwords*
- La implementación de lowercasing en python resulta realmente sencillo. Esto debido a que los *string* en python cuentan con la propiedad ***lower***, la cual retorna en minúsculas el *string*.

In [10]:
sentence = "Esta ORACIÓN será LLEVADA a MINÚSCULAS"
print(f"Sentencia inicial: {sentence}")
sentence = sentence.lower()
print(f"Sentencia procesada: {sentence}")

Sentencia inicial: Esta ORACIÓN será LLEVADA a MINÚSCULAS
Sentencia procesada: esta oración será llevada a minúsculas


- Para el filtrado de *stopwords* nos podemos apoyar de *nltk*

In [13]:
from nltk.corpus import stopwords
nltk.download('stopwords')
import numpy as np

stop_words = stopwords.words("spanish")
print(stop_words)
print(f"Cinco stopwords: {np.random.choice(stop_words, size=5)}")

['de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un', 'para', 'con', 'no', 'una', 'su', 'al', 'lo', 'como', 'más', 'pero', 'sus', 'le', 'ya', 'o', 'este', 'sí', 'porque', 'esta', 'entre', 'cuando', 'muy', 'sin', 'sobre', 'también', 'me', 'hasta', 'hay', 'donde', 'quien', 'desde', 'todo', 'nos', 'durante', 'todos', 'uno', 'les', 'ni', 'contra', 'otros', 'ese', 'eso', 'ante', 'ellos', 'e', 'esto', 'mí', 'antes', 'algunos', 'qué', 'unos', 'yo', 'otro', 'otras', 'otra', 'él', 'tanto', 'esa', 'estos', 'mucho', 'quienes', 'nada', 'muchos', 'cual', 'poco', 'ella', 'estar', 'estas', 'algunas', 'algo', 'nosotros', 'mi', 'mis', 'tú', 'te', 'ti', 'tu', 'tus', 'ellas', 'nosotras', 'vosotros', 'vosotras', 'os', 'mío', 'mía', 'míos', 'mías', 'tuyo', 'tuya', 'tuyos', 'tuyas', 'suyo', 'suya', 'suyos', 'suyas', 'nuestro', 'nuestra', 'nuestros', 'nuestras', 'vuestro', 'vuestra', 'vuestros', 'vuestras', 'esos', 'esas', 'estoy', 'estás', 'está', 'estamos', 'estáis', 'están', 'e

[nltk_data] Downloading package stopwords to /home/edwin/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [14]:
len(stop_words)

313

- Cuando tokenizamos, podemos filtrar nuestras palabras para eliminar las *stopwords*

In [16]:
tokens = sentence.split()
filter_tokens = [token for token in tokens if token not in stop_words]
print(f"sentencia inicial: {sentence}")
print(f"sentenccia procesada {filter_tokens}")

sentencia inicial: esta oración será llevada a minúsculas
sentenccia procesada ['oración', 'llevada', 'minúsculas']


### Signos de puntuación o especiales

En algunos casos los signos de puntuación agregan ruido a nuestro corpus, aumentando su complejidad. Aunque es clave en casos como el análisis de sentimientos.

##### ¿Cómo podemos eliminar signos de puntución o caracteres especiales?

1 -> {"a": "b"}  2 -> ('acd', 'bef'), ('b', '1', 'a')

In [18]:
import string

sentence = "* Hola, esta sentencia tiene puntuación!!!"
punc = string.punctuation
print(f"signos: {punc}")
print(f"sentencia: {sentence}")
mk_trans = str.maketrans('', '', punc)
sentence.translate(mk_trans)

signos: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
sentencia: * Hola, esta sentencia tiene puntuación!!!


' Hola esta sentencia tiene puntuación'

Usando [expresiones regulares](https://docs.python.org/es/3.13/library/re.html)

In [19]:
import re

re.sub(r'[^a-zA-Z0-9\s]','', sentence)

' Hola esta sentencia tiene puntuacin'

### Apliquemos lo visto al corpus CESS-ESP
Para este ejemplo, solo cargaremos las sentencias del corpus, sin su etiqueta morfológica. Lo que haremos es dos filtrados.
1) Filtrado basado en lematización:
    - Para el filtrado, haremos los siguiente:
      1) Aplicaremos lowercassing a todas las palabras
      2) Eliminaremos las *stopwords*
      3) Lematizamos todos los tokens
<br><br>
2) Filtrado basado en stemmer:
   - procesaremos así:
     1) lowercassing a todas las palabras
     2) Eliminación de *stopwrods*
     3) stemmer de los tokens
    

Al final, comparamos reducción de dimensionalidad y tiempos de ejecución del procesamiento

- Definimos una función para cargar los datos

In [20]:
from nltk.corpus import cess_esp

init_words = cess_esp.words()
init_len = len(np.unique(init_words))
print(f"vocabulario inicial {init_len}")

vocabulario inicial 25464


In [21]:
def preprocesing(words):
    pre_words = []
    for w in words:
        pre_w = w.lower()
        pre_w = pre_w.translate(mk_trans)
        if len(pre_w) > 1 and pre_w not in stop_words:
            pre_words.append(pre_w)
    return pre_words

In [23]:
from time import time

tic = time()
post_data = preprocesing(init_words)
toc = time()

In [24]:
post_len = len(np.unique(post_data))
print(f"vocabulario despues eliminar puntuación y lowercasing {post_len}")

vocabulario despues eliminar puntuación y lowercasing 24092


In [25]:
for i in range(20):
    print(f"plabra inicial: {init_words[i]:10} -> {post_data[i]}")

plabra inicial: El         -> grupo
plabra inicial: grupo      -> estatal
plabra inicial: estatal    -> electricitédefrance
plabra inicial: Electricité_de_France -> fpa
plabra inicial: -Fpa-      -> edf
plabra inicial: EDF        -> fpt
plabra inicial: -Fpt-      -> anunció
plabra inicial: anunció    -> hoy
plabra inicial: hoy        -> jueves
plabra inicial: ,          -> compra
plabra inicial: jueves     -> 51porciento
plabra inicial: ,          -> empresa
plabra inicial: la         -> mexicana
plabra inicial: compra     -> electricidadáguiladealtamira
plabra inicial: del        -> fpa
plabra inicial: 51_por_ciento -> eaa
plabra inicial: de         -> fpt
plabra inicial: la         -> creada
plabra inicial: empresa    -> japonés
plabra inicial: mexicana   -> mitsubishicorporation


##### Filtrado basado en lematización

In [26]:
nlp_pipe = spacy.load("es_core_news_sm") 

def apply_lemma_filter(tokens):
    lemmas = []
    count = 0
    tic = time()
    for token in tokens:
        count += 1
        lemma = nlp_pipe(str(token)).doc[0].lemma_
        lemmas.append(lemma)
        if count%1000 == 0:
            toc = time()
            print(f"procesado {count} time: {toc-tic} [s]")
            tic = time()
    return lemmas

In [27]:
unique_post_words = list(np.unique(post_data))
unique_post_words[:10]

['00',
 '01',
 '02',
 '0242',
 '03',
 '035porciento',
 '0420001600',
 '0420040500',
 '0420040800',
 '05']

In [28]:
from time import time

tic = time()
lemma_words = apply_lemma_filter(unique_post_words)
toc = time()
print(f"tiempo ejecución: {(toc-tic)/60} [min]")

procesado 1000 time: 6.736114740371704 [s]
procesado 2000 time: 6.584475517272949 [s]
procesado 3000 time: 6.701909065246582 [s]
procesado 4000 time: 8.091142416000366 [s]
procesado 5000 time: 7.3894431591033936 [s]
procesado 6000 time: 7.120740175247192 [s]
procesado 7000 time: 6.778027057647705 [s]
procesado 8000 time: 6.797396183013916 [s]
procesado 9000 time: 6.827404260635376 [s]
procesado 10000 time: 7.996314525604248 [s]
procesado 11000 time: 8.36932897567749 [s]
procesado 12000 time: 8.07861852645874 [s]
procesado 13000 time: 8.114355564117432 [s]
procesado 14000 time: 7.894568681716919 [s]
procesado 15000 time: 8.035095691680908 [s]
procesado 16000 time: 7.01251482963562 [s]
procesado 17000 time: 7.028913259506226 [s]
procesado 18000 time: 6.9660584926605225 [s]
procesado 19000 time: 6.974210500717163 [s]
procesado 20000 time: 7.034407377243042 [s]
procesado 21000 time: 7.260646343231201 [s]
procesado 22000 time: 7.1397035121917725 [s]
procesado 23000 time: 7.0225114822387695 

##### Filtro basado en stemmer

In [29]:
stemmer = SnowballStemmer('spanish')

def apply_stemmer_filter(tokens):
    stemmers = []
    count = 0 
    for token in tokens:
        count += 1
        stem = stemmer.stem(token)
        stemmers.append(stem)
        if count%1000 == 0:
            print(f"procesado {count}")
    return stemmers

In [30]:
from time import time

tic = time()
stemmer_words = apply_stemmer_filter(post_data)
toc = time()
print(f"tiempo ejecución: {(toc-tic)/60} [min]")

procesado 1000
procesado 2000
procesado 3000
procesado 4000
procesado 5000
procesado 6000
procesado 7000
procesado 8000
procesado 9000
procesado 10000
procesado 11000
procesado 12000
procesado 13000
procesado 14000
procesado 15000
procesado 16000
procesado 17000
procesado 18000
procesado 19000
procesado 20000
procesado 21000
procesado 22000
procesado 23000
procesado 24000
procesado 25000
procesado 26000
procesado 27000
procesado 28000
procesado 29000
procesado 30000
procesado 31000
procesado 32000
procesado 33000
procesado 34000
procesado 35000
procesado 36000
procesado 37000
procesado 38000
procesado 39000
procesado 40000
procesado 41000
procesado 42000
procesado 43000
procesado 44000
procesado 45000
procesado 46000
procesado 47000
procesado 48000
procesado 49000
procesado 50000
procesado 51000
procesado 52000
procesado 53000
procesado 54000
procesado 55000
procesado 56000
procesado 57000
procesado 58000
procesado 59000
procesado 60000
procesado 61000
procesado 62000
procesado 63000
p

### Conclusiones

- El aplicar técnicas de NLP nos permite reducir dimensionalidad de nuestros datos, lo que disminuye el tiempo de entrenamiento de modelos de AI.
- También nos permite eliminar palabras que no otorgan importancia en nuestro corpus.
- No existe un receta única para el procesado del corpus, depende de varios factores.
- Si de puede desistir del significado real de cada palabra, utilizar *stemming* reduce el tiempo de procesamiento del corpus.

#### Requisitos:
- Spacy 3.7.0

### Documentación
- https://www.nltk.org/index.html
- https://snowballstem.org/algorithms/spanish/stemmer.html